### 3.3. Load dataset
In our curves of Demand and Supply graph, the energy load represents the Demand.  
In this final part of data cleaning, we will follow the same cleaning framework as the previous ones:
1. The aggregation of the five files containing TERNA data about generated electricity so that we have the complete time series from 2021 to 2025;
2. The deletetion of rows with NULL values, due to the final message that occupies only 1 or 2 cells on each Excel file, indicating which filter had been applied while manually downloading the data;
3. The assignment of the correct data type to each column;
4. Mtu (Market Time Unit) harmonization.

But before executing the Market Time Unit harmonization, this process includes the creation of two datasets:  
I. The first one will be used for national comparison to the Supply and Prices data;  
II. The second one, is created for unnecessary analysis that could still provide valuable insights on zonal consumption trends.

In [1]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Future Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

# define initial variables
dataset_path = Path('../dataset/raw')           # input folder path
output_path = Path('../dataset/clean')          # output folder path
output_path.mkdir(parents=True, exist_ok=True)  # create folder if non-existent

YEARS = range(2021, 2026)

In [3]:
# 1. Aggregate load data in dataframe
load_dfs = [
    pd.read_excel(dataset_path / f'TERNA-load-{year}.xlsx')
    for year in YEARS
]
load_data = pd.concat(load_dfs, ignore_index=True)

# fast check
print(f'''
----Rows x Cols:----
{load_data.shape}\n
----First rows:----
{load_data.head()}\n
----Last rows:----
{load_data.tail()}
''')


----Rows x Cols:----
(1402372, 4)

----First rows:----
                  Date  Total Load [MW]  Forecast Total Load [MW]  \
0  2021-12-31 23:45:00          515.767                   516.717   
1  2021-12-31 23:45:00         2139.978                  2143.918   
2  2021-12-31 23:45:00         4678.492                  4687.105   
3  2021-12-31 23:45:00        12240.020                 12262.554   
4  2021-12-31 23:45:00          883.006                   884.631   

   Bidding Zone  
0      Calabria  
1  Centre-North  
2  Centre-South  
3         North  
4      Sardinia  

----Last rows:----
                                    Date  Total Load [MW]  \
1402367              2025-01-01 00:00:00         4621.470   
1402368              2025-01-01 00:00:00         2115.609   
1402369              2025-01-01 00:00:00          530.625   
1402370              2025-01-01 00:00:00        24928.999   
1402371  Applied filters:  Year is 2025;              NaN   

         Forecast Total Load [MW] 

In [5]:
# check
print(f'''
{load_data.info()}\n
----Date description----
{load_data['Date'].describe()}\n
----Actual energy load description----
{load_data['Total Load [MW]'].describe()}\n
----Forecast energy load description----
{load_data['Forecast Total Load [MW]'].describe()}\n
----Source of production----
{load_data['Bidding Zone'].describe()}\n
''')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1402372 entries, 0 to 1402371
Data columns (total 4 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Date                      1402372 non-null  object 
 1   Total Load [MW]           1402367 non-null  float64
 2   Forecast Total Load [MW]  1402367 non-null  float64
 3   Bidding Zone              1402367 non-null  object 
dtypes: float64(2), object(2)
memory usage: 42.8+ MB

None

----Date description----
count                 1402372
unique                 175281
top       2023-10-29 02:15:00
freq                       16
Name: Date, dtype: object

----Actual energy load description----
count    1.402367e+06
mean     8.918439e+03
std      1.221567e+04
min     -7.435000e+01
25%      1.367441e+03
50%      2.844375e+03
75%      9.791538e+03
max      5.865100e+04
Name: Total Load [MW], dtype: float64

----Forecast energy load description----
count    1.402367e+

In [6]:
# 2. Delete n=5 Rows with NULL values it's the row at the end of each dataset (for each year) which explains the applied filter
load_data = load_data.dropna(axis=0, how='any')
load_data.head(10)       # check

,Date,Total Load [MW],Forecast Total Load [MW],Bidding Zone
0,2021-12-31 23:45:00,515.767,516.717,Calabria
1,2021-12-31 23:45:00,2139.978,2143.918,Centre-North
2,2021-12-31 23:45:00,4678.492,4687.105,Centre-South
3,2021-12-31 23:45:00,12240.020,12262.554,North
4,2021-12-31 23:45:00,883.006,884.631,Sardinia
5,2021-12-31 23:45:00,1863.289,1866.719,Sicily
6,2021-12-31 23:45:00,2122.448,2126.355,South
7,2021-12-31 23:45:00,24443.000,24487.999,Italy
8,2021-12-31 23:30:00,24854.000,24715.000,Italy
9,2021-12-31 23:30:00,526.782,523.836,Calabria


In [7]:
# 3. Convert Date from Object type to Datetime type
load_data['Date'] = pd.to_datetime(load_data['Date'], dayfirst=True, errors='coerce')
#    Convert Bidding Zone from Object to String
load_data['Bidding Zone'] = load_data['Bidding Zone'].astype('string')

From this dataset, it is helpful to create two separate tables:
1. The first one only selects the Italy column which is already provided by TERNA and it serves for analyzing the total load in Italy;
2. The second one, which details the bidding zone, could be helpful for zonal analysis.
This distinction has been implemented for optimizing hard disk space and time analysis, since the merged dataset is more than 1mln rows long.

In [21]:
# 1. Italy dataset
load_italy = load_data[load_data['Bidding Zone'] == 'Italy'].copy()

load_italy = (
    load_italy
    .groupby(pd.Grouper(key='Date', freq='h'))[['Total Load [MW]', 'Forecast Total Load [MW]']]
    .mean()
    .reset_index()
)

load_italy['Total Load [MW]'] = load_italy['Total Load [MW]'].clip(lower=0)

# rename columns
load_italy = load_italy.rename(columns={
    'Total Load [MW]': 'Total Load',
    'Forecast Total Load [MW]': 'Load forecasted'
})

load_italy

,Date,Total Load,Load forecasted
0,2021-01-01 00:00:00,24640.25075,24614.24975
1,2021-01-01 01:00:00,22909.00000,23971.99975
2,2021-01-01 02:00:00,21044.49975,22554.74950
3,2021-01-01 03:00:00,19907.99975,21153.49975
4,2021-01-01 04:00:00,19578.49950,20336.75000
...,...,...,...
43819,2025-12-31 19:00:00,37002.25000,36421.74925
43820,2025-12-31 20:00:00,34212.50000,32732.99975
43821,2025-12-31 21:00:00,31110.24950,29597.99925
43822,2025-12-31 22:00:00,29077.25000,27451.50025


In [14]:
# 2. Zonal dataset
load_zones = load_data[load_data['Bidding Zone'] != 'Italy'].copy()

load_zones = (
    load_zones
    .groupby(['Bidding Zone', pd.Grouper(key='Date', freq='h')])['Total Load [MW]']
    .mean()
    .reset_index()
)

load_zones_pivot = load_zones.pivot(
    index='Date',
    columns='Bidding Zone',
    values='Total Load [MW]'
)

load_zones_pivot

Bidding Zone,Calabria,Centre-North,Centre-South,North,Sardinia,Sicily,South
Date,,,,,,,
2021-01-01 00:00:00,578.580667,2395.75825,4680.37000,11979.33375,978.29950,1950.45250,2222.10125
2021-01-01 01:00:00,533.948750,2184.13725,4441.81075,11085.10825,895.01900,1835.80450,1933.17150
2021-01-01 02:00:00,474.600750,2031.84300,4050.46325,10197.47000,832.86850,1665.08025,1792.17400
2021-01-01 03:00:00,431.402750,1951.77700,3748.25950,9775.42225,794.61650,1535.85550,1670.66625
2021-01-01 04:00:00,405.532500,1900.44950,3611.61400,9726.83750,791.27075,1497.10275,1645.69250
...,...,...,...,...,...,...,...
2025-12-31 19:00:00,917.221250,3240.59375,7001.60650,18576.11425,1169.87050,2967.73700,3129.10675
2025-12-31 20:00:00,832.167500,3019.89850,6355.46350,17223.44825,1090.23475,2702.20925,2989.07825
2025-12-31 21:00:00,813.606000,2732.36975,5644.37825,15822.54725,998.01500,2368.82350,2730.50975


After creating two separate datasets for two different aims, it is advisable to export both datasets into the output folder.

In [22]:
# export new datasets
load_italy.to_csv(output_path / 'load-italy-clean.csv', index=False)
load_zones_pivot.to_csv(output_path / 'load-zones-clean.csv')

Back: [3.2. Energy generation data cleaning](3.2_gen-data-cleaning.ipynb)  
Next: [3.4. Dataset merge](3.4_dataset-merge.ipynb)